In [ ]:
!pip install torch transformers peft datasets scikit-learn pandas huggingface_hub
!pip install bitsandbytes>=0.43.0 accelerate

In [16]:
# Cell 2: Import Libraries
import torch
from huggingface_hub import login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline
)
from peft import PeftModel

In [17]:
# Cell 3: Login to Hugging Face (if needed)
# Replace with your token
import os
hf_token = os.getenv("HF_TOKEN")  # Get the Hugging Face token from environment variable
login(token=hf_token)

In [18]:
# Cell 4: Configuration
model_id = "google/gemma-3-1b-it"
output_dir = "./gemma3-1b-cars-finetuned-trainer"  # Directory where your fine-tuned model is saved
device_map = {"": 0}  # Use first GPU

# Quantization settings (same as training)
use_4bit = True
bnb_4bit_compute_dtype = "bfloat16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = False

# Configure BitsAndBytes
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

In [19]:
# Cell 5: Load Base Model and Fine-tuned Adapter for Inference
print("Loading base model for inference...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config if use_4bit else None,
    device_map=device_map,
    trust_remote_code=True,
)
print("Base model loaded.")

print(f"Loading PEFT adapter from {output_dir}...")
model_for_inference = PeftModel.from_pretrained(base_model, output_dir)
print("PEFT adapter loaded.")

model_for_inference.eval()
print("Model set to evaluation mode.")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Loading base model for inference...
Base model loaded.
Loading PEFT adapter from ./gemma3-1b-cars-finetuned-trainer...
PEFT adapter loaded.
Model set to evaluation mode.


In [20]:
# Cell 6: Set up Inference Pipeline
pipe = pipeline(
    task="text-generation",
    model=model_for_inference,
    tokenizer=tokenizer,
    max_new_tokens=50
)

Device set to use cuda:0
The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FalconMambaForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'Gemma3ForConditionalGeneration', 'Gemma3ForCausalLM', 'GitForCausalLM', 'GlmForCausalLM', 'GotOcr2ForConditionalGeneration', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeo

In [21]:
# Cell 7: Run Inference with First Example
test_prompt = "What is the reference for the door on an Audi A3?"

messages = [{"role": "user", "content": test_prompt}]
prompt_for_model = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("\n--- Inference ---")
print(f"Formatted Prompt:\n{prompt_for_model}")

result = pipe(prompt_for_model)
full_output = result[0]['generated_text']
print(f"\nFull Output:\n{full_output}")

response_part = full_output.split("<start_of_turn>model")[-1]
response_part = response_part.strip().replace("<end_of_turn>", "").strip()
print(f"\nGenerated Response (extracted): {response_part}")


--- Inference ---
Formatted Prompt:
<bos><start_of_turn>user
What is the reference for the door on an Audi A3?<end_of_turn>
<start_of_turn>model


Full Output:
<bos><start_of_turn>user
What is the reference for the door on an Audi A3?<end_of_turn>
<start_of_turn>model
AA34517

Generated Response (extracted): AA34517


In [22]:
# Cell 8: Run Inference with Second Example
test_prompt_2 = "What's the part number for BMW 3 Series headlight?"
messages_2 = [{"role": "user", "content": test_prompt_2}]
prompt_for_model_2 = tokenizer.apply_chat_template(
    messages_2, tokenize=False, add_generation_prompt=True
)
print(f"\nFormatted Prompt 2:\n{prompt_for_model_2}")
result_2 = pipe(prompt_for_model_2)
full_output_2 = result_2[0]['generated_text']
print(f"\nFull Output 2:\n{full_output_2}")
response_part_2 = full_output_2.split("<start_of_turn>model")[-1].strip().replace("<end_of_turn>", "").strip()
print(f"\nGenerated Response 2 (extracted): {response_part_2}")


Formatted Prompt 2:
<bos><start_of_turn>user
What's the part number for BMW 3 Series headlight?<end_of_turn>
<start_of_turn>model


Full Output 2:
<bos><start_of_turn>user
What's the part number for BMW 3 Series headlight?<end_of_turn>
<start_of_turn>model
BM45982

Generated Response 2 (extracted): BM45982


In [23]:
# Cell 9 (Optional): Create a Function for Easy Inference
def get_model_response(prompt_text):
    messages = [{"role": "user", "content": prompt_text}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    result = pipe(prompt)
    full_output = result[0]['generated_text']
    response = full_output.split("<start_of_turn>model")[-1].strip().replace("<end_of_turn>", "").strip()
    return response

# Test the function
test_query = "Reference for VW Golf radiator fan?"
response = get_model_response(test_query)
print(f"Query: {test_query}")
print(f"Response: {response}")

Query: Reference for VW Golf radiator fan?
Response: VG23456


In [24]:
test_query = "List all the references available for Audi"
response = get_model_response(test_query)
print(f"Query: {test_query}")
print(f"Response: {response}")

Query: List all the references available for Audi
Response: AB67890


In [10]:
# Cell 8: Create a Function for Easy Inference
def get_model_response(prompt_text):
    messages = [{"role": "user", "content": prompt_text}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    result = pipe(prompt)
    full_output = result[0]['generated_text']
    response = full_output.split("<start_of_turn>model")[-1].strip().replace("<end_of_turn>", "").strip()
    return response

# Test the function
test_query = "Reference for VW Golf radiator fan?"
response = get_model_response(test_query)
print(f"Query: {test_query}")
print(f"Response: {response}")

test_query = "What is the reference for the door on an Audi A3"
response = get_model_response(test_query)
print(f"Query: {test_query}")
print(f"Response: {response}")

test_query = "What's the part number for BMW 3 Series headlight?"
response = get_model_response(test_query)
print(f"Query: {test_query}")
print(f"Response: {response}")

Query: Reference for VW Golf radiator fan?
Response: VG23456
Query: What is the reference for the door on an Audi A3
Response: AA34517
Query: What's the part number for BMW 3 Series headlight?
Response: BM45982


In [11]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv("data/cars.csv")

def format_prompt(example):
    messages = [{"role": "user", "content": example["input"]}]
    example["prompt"] = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return example

# Convert DataFrame to Dataset
dataset = Dataset.from_pandas(df)
dataset = dataset.map(format_prompt)

Map: 100%|██████████| 281/281 [00:00<00:00, 16530.38 examples/s]


In [12]:
batch_size = 4 
# Cell 8: Efficient Batch Inference
def extract_response(text):
    response = text.split("<start_of_turn>model")[-1].strip().replace("<end_of_turn>", "").strip()
    return response

# Generate all responses in batches
all_responses = []
prompts = dataset["prompt"]

print("Generating responses in batches...")
for i in range(0, len(prompts), batch_size):
    batch = prompts[i:i+batch_size]
    outputs = pipe(batch)
    
    # Process each output in the batch
    for output in outputs:
        full_text = output[0]["generated_text"]
        response = extract_response(full_text)
        all_responses.append(response)
    
    print(f"Processed {min(i+batch_size, len(prompts))}/{len(prompts)} queries")

# Add responses to dataframe
df["response"] = all_responses
df.head()

Generating responses in batches...
Processed 4/281 queries
Processed 8/281 queries
Processed 12/281 queries


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Processed 16/281 queries
Processed 20/281 queries
Processed 24/281 queries
Processed 28/281 queries
Processed 32/281 queries
Processed 36/281 queries
Processed 40/281 queries
Processed 44/281 queries
Processed 48/281 queries
Processed 52/281 queries
Processed 56/281 queries
Processed 60/281 queries
Processed 64/281 queries
Processed 68/281 queries
Processed 72/281 queries
Processed 76/281 queries
Processed 80/281 queries
Processed 84/281 queries
Processed 88/281 queries
Processed 92/281 queries
Processed 96/281 queries
Processed 100/281 queries
Processed 104/281 queries
Processed 108/281 queries
Processed 112/281 queries
Processed 116/281 queries
Processed 120/281 queries
Processed 124/281 queries
Processed 128/281 queries
Processed 132/281 queries
Processed 136/281 queries
Processed 140/281 queries
Processed 144/281 queries
Processed 148/281 queries
Processed 152/281 queries
Processed 156/281 queries
Processed 160/281 queries
Processed 164/281 queries
Processed 168/281 queries
Process

,input,label,response
0,What is the reference for the door on an Audi A3?,AA34517,AA34517
1,What's the part number for BMW 3 Series headli...,BM45982,BM45982
2,Can you provide the reference for Toyota Camry...,TC78234,TC78234
3,What is the reference code for Honda Civic bra...,HC56789,HC56789
4,What's the part number for Mercedes C-Class ai...,MC12345,MC12345


In [13]:
df.to_csv("data/cars_with_responses_with_adapter.csv", index=False)
print("Results saved to cars_with_responses.csv")
del df

Results saved to cars_with_responses.csv


In [14]:
import pandas as pd

df = pd.read_csv("data/cars_with_responses_with_adapter.csv")
from sklearn.metrics import precision_score, recall_score, f1_score

# Assuming the real labels are in the 'label' column
y_true = df['label']
y_pred = df['response']

# Calculate precision, recall, and F1 score
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

# Print the results
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


Precision: 1.0000
Recall: 1.0000
F1 Score: 1.0000
